In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
import re
warnings.filterwarnings('ignore')

In [2]:
def make_features(df):
    agg_funcs = ['min', 'max', 'mean', 'std', 'sum', 'median']
    feats = df.groupby('id').agg({
        'down_time': agg_funcs,
        'up_time': agg_funcs,
        'action_time': agg_funcs,
        'cursor_position': agg_funcs,
        'word_count': agg_funcs
    })
    feats.columns = ['_'.join(col) for col in feats.columns]
    
    act_counts = df.groupby(['id', 'activity']).size().unstack(fill_value=0)
    act_counts.columns = [f'act_{c}' for c in act_counts.columns]
    feats = feats.join(act_counts)
    
    top_events = ['q', 'Space', 'Backspace', 'Shift', 'ArrowRight', 'Leftclick', 'ArrowLeft', 'period', 'comma', 'Enter']
    event_mapping = {'.': 'period', ',': 'comma'}
    
    df['down_event_mapped'] = df['down_event'].replace(event_mapping)
    for event in top_events:
        feats[f'ev_{event}'] = df[df['down_event_mapped'] == event].groupby('id').size()
        
    feats = feats.fillna(0)
    
 
    feats.columns = [re.sub(r'[^A-Za-z0-9_]', '_', str(c)) for c in feats.columns]
    
    return feats

In [3]:
train_logs = pd.read_csv('train_logs.csv')
train_scores = pd.read_csv('train_scores.csv')
test_logs = pd.read_csv('test_logs.csv')

X = make_features(train_logs)
y = train_scores.set_index('id').loc[X.index, 'score']
X_test = make_features(test_logs)

X_test = X_test.reindex(columns=X.columns, fill_value=0)


In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
preds = np.zeros(len(X_test))
oof = np.zeros(len(X))

for train_idx, val_idx in kf.split(X):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500, 
        learning_rate=0.02, 
        max_depth=5, 
        colsample_bytree=0.7, 
        random_state=42
    )
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    model_cb = CatBoostRegressor(
        iterations=1500, 
        learning_rate=0.02, 
        depth=5, 
        random_seed=42, 
        verbose=0
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)
    
    oof[val_idx] = (model_lgb.predict(X_va) + model_cb.predict(X_va)) / 2
    preds += ((model_lgb.predict(X_test) + model_cb.predict(X_test)) / 2) / kf.n_splits

print(f'OOF RMSE: {np.sqrt(mean_squared_error(y, oof))}')

sub = pd.DataFrame({'id': X_test.index, 'score': preds})
sub.to_csv('submission.csv', index=False)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001386 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9137
[LightGBM] [Info] Number of data points in the train set: 2223, number of used features: 43
[LightGBM] [Info] Start training from score 3.706928
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 